In [1]:
# Install Bioconductor + required packages
if (!requireNamespace("BiocManager", quietly = TRUE))
  install.packages("BiocManager")

BiocManager::install(c("ensembldb", "AnnotationHub"), ask = FALSE, update = FALSE)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

'getOption("repos")' replaces Bioconductor standard repositories, see
'help("repositories", package = "BiocManager")' for details.
Replacement repositories:
    CRAN: https://cran.rstudio.com

Bioconductor version 3.23 (BiocManager 1.30.27), R 4.6.0 (2026-04-24)

Installing package(s) 'BiocVersion', 'ensembldb', 'AnnotationHub'

also installing the dependencies ‘matrixStats’, ‘abind’, ‘SparseArray’, ‘formatR’, ‘MatrixGenerics’, ‘S4Arrays’, ‘DelayedArray’, ‘lambda.r’, ‘futile.options’, ‘png’, ‘SummarizedExperiment’, ‘cigarillo’, ‘RCurl’, ‘rjson’, ‘futile.logger’, ‘snow’, ‘BH’, ‘XVector’, ‘lazyeval’, ‘UCSC.utils’, ‘KEGGREST’, ‘XML’, ‘GenomicAlignments’, ‘BiocIO’, ‘restfulr’, ‘bitops’, ‘BiocParallel’, ‘Rhtslib’, ‘filelock’, ‘BiocGenerics’, ‘GenomicRanges’, ‘GenomicFeatures’, ‘AnnotationFilter’, ‘RSQLite’, ‘Biobase’, ‘Seqinfo’, ‘GenomeInfoDb’, ‘AnnotationDbi’, ‘rtracklayer’, ‘S4Vectors’, ‘Rsamtools’, ‘IRange

In [2]:
# Load libraries and connect to EnsDb
library(ensembldb)
library(AnnotationHub)

# Connect to AnnotationHub
ah <- AnnotationHub()

# AH119325 = Ensembl 110, Homo sapiens
edb <- ah[["AH119325"]]

# Quick check — confirm it loaded
edb

Loading required package: BiocGenerics

Loading required package: generics


Attaching package: ‘generics’


The following objects are masked from ‘package:base’:

    as.difftime, as.factor, as.ordered, intersect, is.element, setdiff,
    setequal, union



Attaching package: ‘BiocGenerics’


The following objects are masked from ‘package:stats’:

    IQR, mad, sd, var, xtabs


The following objects are masked from ‘package:base’:

    anyDuplicated, aperm, append, as.data.frame, basename, cbind,
    colnames, dirname, do.call, duplicated, eval, evalq, Filter, Find,
    get, grep, grepl, is.unsorted, lapply, Map, mapply, match, mget,
    order, paste, pmax, pmax.int, pmin, pmin.int, Position, rank,
    rbind, Reduce, rownames, sapply, saveRDS, table, tapply, unique,
    unsplit, which.max, which.min


Loading required package: GenomicRanges

Loading required package: stats4

Loading required package: S4Vectors


Attaching package: ‘S4Vectors’


The following object is masked from ‘pac

downloading 1 resources

retrieving 1 resource



loading from cache



EnsDb for Ensembl:
|Backend: SQLite
|Db type: EnsDb
|Type of Gene ID: Ensembl Gene ID
|Supporting package: ensembldb
|Db created by: ensembldb package from Bioconductor
|script_version: 0.3.10
|Creation time: Sat Oct 26 21:34:14 2024
|ensembl_version: 113
|ensembl_host: 127.0.0.1
|Organism: Homo sapiens
|taxonomy_id: 9606
|genome_build: GRCh38
|DBSCHEMAVERSION: 2.2
|common_name: human
|species: homo_sapiens
| No. of genes: 87726.
| No. of transcripts: 413674.
|Protein data available.

In [3]:
# ALL HUMAN GENES (ENSG)
all_human_genes <- genes(edb, return.type = "DataFrame")
all_gene_ids    <- all_human_genes$gene_id          # ENSG IDs

cat("Total human genes:", length(all_gene_ids), "\n")

write.table(all_gene_ids,
            file      = "all_human_genes.tsv",
            sep       = "\t",
            row.names = FALSE,
            col.names = FALSE,
            quote     = FALSE)

Total human genes: 87726 


In [ ]:
#  ALL HUMAN TRANSCRIPTS (ENST)
all_human_tx <- transcripts(edb, return.type = "DataFrame")
all_tx_ids   <- all_human_tx$tx_id                  # ENST IDs

cat("Total human transcripts:", length(all_tx_ids), "\n")

write.table(all_tx_ids,
            file      = "all_human_transcripts.tsv",
            sep       = "\t",
            row.names = FALSE,
            col.names = FALSE,
            quote     = FALSE)

In [ ]:
# ALL HUMAN PROTEINS (ENSP)
all_human_proteins <- proteins(edb, return.type = "DataFrame")
all_protein_ids    <- all_human_proteins$protein_id  # ENSP IDs

cat("Total human proteins:", length(all_protein_ids), "\n")

write.table(all_protein_ids,
            file      = "all_human_proteins.tsv",
            sep       = "\t",
            row.names = FALSE,
            col.names = FALSE,
            quote     = FALSE)

In [ ]:
#  HLA-A Transcripts (ENST)
HLAA_gene_id <- "ENSG00000206503"

# Filter transcripts to HLA-A only
hlaa_transcripts <- transcripts(
  edb,
  filter      = GeneIdFilter(HLAA_gene_id),
  return.type = "DataFrame"
)

# Display
cat("=== Transcripts for HLA-A ===\n")
print(hlaa_transcripts[, c("tx_id", "tx_biotype", "tx_seq_start",
                            "tx_seq_end", "tx_cds_seq_start", "tx_cds_seq_end")])

cat("\nTotal transcripts:", nrow(hlaa_transcripts), "\n")

# Save to file
write.table(hlaa_transcripts,
            file      = "HLAA_transcripts.tsv",
            sep       = "\t",
            row.names = FALSE,
            quote     = FALSE)

In [ ]:
#  HLA-A Proteins (ENSP)
hlaa_proteins <- proteins(
  edb,
  filter      = GeneIdFilter(HLAA_gene_id),
  return.type = "DataFrame"
)

# Display
cat("=== Proteins for HLA-A ===\n")
print(hlaa_proteins[, c("protein_id", "tx_id", "protein_sequence")])

cat("\nTotal protein-coding transcripts:", nrow(hlaa_proteins), "\n")

# Save to file
write.table(hlaa_proteins,
            file      = "HLAA_proteins.tsv",
            sep       = "\t",
            row.names = FALSE,
            quote     = FALSE)

In [ ]:
# protein ID stored as IRanges name

library(IRanges)

hlaa_protein_ids <- hlaa_proteins$protein_id
all_maps <- list()

for (prot_id in hlaa_protein_ids) {
  cat("\nProcessing:", prot_id, "\n")

  prot_seq <- hlaa_proteins$protein_sequence[hlaa_proteins$protein_id == prot_id]

  if (is.na(prot_seq) || nchar(prot_seq) == 0) {
    cat("  ⚠ No sequence — skipping\n")
    next
  }

  prot_len <- nchar(prot_seq)
  cat("  Protein length:", prot_len, "aa\n")

  # Set the protein ID as the NAME of the IRanges object
  prot_range <- IRanges(start = 1, end = prot_len, names = prot_id)

  # id = "name" tells the function to look in names(x)
  tryCatch({
    genome_map <- proteinToGenome(x = prot_range, db = edb, id = "name")

    gr <- genome_map[[1]]

    if (length(gr) == 0) {
      cat("  ⚠ No mapping returned\n")
      next
    }

    df <- as.data.frame(gr)
    df$protein_id <- prot_id
    df$prot_length <- prot_len

    all_maps[[prot_id]] <- df
    cat("  ✓ Mapped", nrow(df), "genomic segments\n")

  }, error = function(e) {
    cat("  ✗ Error:", conditionMessage(e), "\n")
  })
}

# Combine & save
if (length(all_maps) > 0) {
  combined_map <- do.call(rbind, all_maps)
  rownames(combined_map) <- NULL

  cat("\n=== Protein-to-Genome Map (first 10 rows) ===\n")
  print(head(combined_map, 10))

  write.table(combined_map,
              file      = "HLAA_protein_to_genome_map.tsv",
              sep       = "\t",
              row.names = FALSE,
              quote     = FALSE)

  cat("\n✓ Saved: HLAA_protein_to_genome_map.tsv\n")
  cat("  Total rows:", nrow(combined_map), "\n")
  cat("  Proteins mapped:", length(all_maps), "of", length(hlaa_protein_ids), "\n")

} else {
  cat("✗ No proteins mapped — try Approach B below\n")
}

In [ ]:
# Inspect the mapping results

# How many proteins mapped successfully?
cat("=== Mapping Summary ===\n")
cat("Total HLA-A proteins:   ", length(hlaa_protein_ids), "\n")
cat("Successfully mapped:    ", length(all_maps), "\n")
cat("Failed / skipped:       ", length(hlaa_protein_ids) - length(all_maps), "\n\n")

# Which ones mapped?
cat("Mapped proteins:\n")
print(names(all_maps))

# Which ones failed?
failed <- setdiff(hlaa_protein_ids, names(all_maps))
if (length(failed) > 0) {
  cat("\nFailed proteins:\n")
  print(failed)
}

# Show the full combined table
cat("\n=== Full Protein-to-Genome Map ===\n")
print(combined_map)

In [ ]:
# Clean per-protein summary

summary_df <- do.call(rbind, lapply(names(all_maps), function(pid) {
  df <- all_maps[[pid]]
  data.frame(
    protein_id      = pid,
    chromosome      = unique(df$seqnames),
    genome_start    = min(df$start),
    genome_end      = max(df$end),
    strand          = unique(df$strand),
    n_exon_segments = nrow(df),
    protein_length  = unique(df$prot_length)
  )
}))

cat("--- Per-Protein Genomic Summary ---\n")
print(summary_df)

write.table(summary_df,
            file      = "HLAA_protein_summary.tsv",
            sep       = "\t",
            row.names = FALSE,
            quote     = FALSE)

cat("\n✓ Saved: HLAA_protein_summary.tsv\n")

In [ ]:
# Transcript → Genome mapping

hlaa_tx_ids <- hlaa_transcripts$tx_id
tx_maps     <- list()

for (tx_id in hlaa_tx_ids) {
  cat("Processing transcript:", tx_id, "\n")

  tryCatch({
    # fetch the actual spliced exon lengths from the database
    exons_for_tx <- exonsBy(edb, by = "tx", filter = TxIdFilter(tx_id))

    # If no exons found, skip
    if (!tx_id %in% names(exons_for_tx)) {
      cat("  ⚠ No exons found — skipping\n")
      next
    }

    # Actual transcript length = sum of all exon widths
    actual_tx_len <- sum(width(exons_for_tx[[tx_id]]))
    cat("  Actual spliced length:", actual_tx_len, "bp\n")

    # Build IRanges in transcript space (1 to actual mRNA length)
    tx_range <- IRanges(start = 1, end = actual_tx_len, names = tx_id)

    tx_map <- transcriptToGenome(x = tx_range, db = edb, id = "name")
    gr     <- tx_map[[1]]

    if (length(gr) == 0) {
      cat("  ⚠ Empty mapping returned\n")
      next
    }

    df          <- as.data.frame(gr)
    df$tx_id    <- tx_id
    df$tx_length <- actual_tx_len
    tx_maps[[tx_id]] <- df
    cat("  ✓ Mapped", nrow(df), "segments\n")

  }, error = function(e) {
    cat("  ✗ Error:", conditionMessage(e), "\n")
  })
}

#  Combine & save
if (length(tx_maps) > 0) {
  tx_combined <- do.call(rbind, tx_maps)
  rownames(tx_combined) <- NULL

  cat("\n=== Transcript-to-Genome Map (first 10 rows) ===\n")
  print(head(tx_combined, 10))

  write.table(tx_combined,
              file      = "HLAA_transcript_to_genome_map.tsv",
              sep       = "\t",
              row.names = FALSE,
              quote     = FALSE)

  cat("\n✓ Saved: HLAA_transcript_to_genome_map.tsv\n")
  cat("  Total rows:", nrow(tx_combined), "\n")
  cat("  Transcripts mapped:", length(tx_maps), "of", length(hlaa_tx_ids), "\n")

} else {
  cat("✗ No transcripts mapped\n")
}

In [ ]:
# List all output files created

output_files <- c(
  "all_human_genes.tsv",
  "all_human_transcripts.tsv",
  "all_human_proteins.tsv",
  "HLAA_transcripts.tsv",
  "HLAA_proteins.tsv",
  "HLAA_protein_to_genome_map.tsv",
  "HLAA_protein_summary.tsv",
  "HLAA_transcript_to_genome_map.tsv"
)

cat("=== Files ready to download ===\n")
for (f in output_files) {
  exists <- file.exists(f)
  size   <- ifelse(exists, paste(file.size(f), "bytes"), "NOT FOUND")
  cat(sprintf("  [%s] %s — %s\n", ifelse(exists, "✓", "✗"), f, size))
}